# 🏟️ Análisis Cinemático de Fútbol — Runner GPU (Colab)

**Pasos antes de ejecutar:**
1. En Colab: `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`
2. Sube `video.mp4` a Google Drive en la ruta: `Mi unidad/futbol_cv/video.mp4`
3. Ejecuta las celdas en orden

In [ ]:
# ── Celda 1: Montar Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Verificar que el video existe en Drive
import os
VIDEO_EN_DRIVE = '/content/drive/MyDrive/futbol_cv/video.mp4'
assert os.path.exists(VIDEO_EN_DRIVE), f'Video no encontrado en {VIDEO_EN_DRIVE}'

In [ ]:
# ── Celda 2: Clonar repositorio e instalar dependencias ──────────────────────
!git clone https://github.com/aldajica-dev/distancia_futbol_CV.git /content/proyecto
%cd /content/proyecto
!pip install -q ultralytics opencv-python-headless numpy pyyaml psutil scikit-learn

In [ ]:
# ── Celda 3: Copiar video al path esperado por el proyecto ───────────────────
import shutil
os.makedirs('/content/proyecto/data/raw', exist_ok=True)
shutil.copy(VIDEO_EN_DRIVE, '/content/proyecto/data/raw/video.mp4')
print('Video copiado:', os.path.getsize('/content/proyecto/data/raw/video.mp4') / 1e6, 'MB')

In [ ]:
# ── Celda 4: Verificar GPU ────────────────────────────────────────────────────
import torch
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── Celda 5: Ejecutar estudio de ablación completo ────────────────────────────
# La salida muestra el progreso de cada experimento
!python /content/proyecto/src/models/advanced_tracker.py

In [ ]:
# ── Celda 6: Copiar resultados a Google Drive ─────────────────────────────────
import shutil

DRIVE_RESULTS = '/content/drive/MyDrive/futbol_cv/resultados'
os.makedirs(DRIVE_RESULTS, exist_ok=True)

PROCESSED = '/content/proyecto/data/processed'
archivos = [
    'ablation_results.json',
    'exp1_metrics.json', 'exp2_metrics.json', 'exp3_metrics.json', 'exp4_metrics.json',
    'exp1_distances.csv', 'exp2_distances.csv', 'exp3_distances.csv', 'exp4_distances.csv',
    'exp1_frame_data.csv', 'exp2_frame_data.csv', 'exp3_frame_data.csv', 'exp4_frame_data.csv',
    'exp4_tactical.mp4',
]

for f in archivos:
    src = os.path.join(PROCESSED, f)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(DRIVE_RESULTS, f))
        print(f'  ✓ {f}')
    else:
        print(f'  ✗ {f} (no encontrado)')

print(f'\nResultados guardados en: {DRIVE_RESULTS}')

In [ ]:
# ── Celda 7 (opcional): Ver tabla comparativa ─────────────────────────────────
import json

with open(os.path.join(PROCESSED, 'ablation_results.json')) as f:
    results = json.load(f)

keys   = ['baseline', 'exp1', 'exp2', 'exp3', 'exp4']
labels = ['Baseline', 'Exp 1', 'Exp 2', 'Exp 3', 'Exp 4']

metricas = [
    ('IDs únicos',           'tracking',          'n_unique_ids'),
    ('Fragmentación (x)',    'tracking',          'fragmentation_ratio'),
    ('ID switches est.',     'tracking',          'id_switches_estimated'),
    ('Track length (fr)',    'tracking',          'avg_track_length_frames'),
    ('Vel. máxima (m/s)',    'kinematics',        'max_speed_ms'),
    ('% vel. imposible',     'kinematics',        'pct_impossible_speed'),
    ('Dist/jugador (m)',     'distances',         'estimated_per_player_m'),
    ('FPS procesamiento',    'computational_cost','fps_processing'),
]

header = f"{'Métrica':<25}" + "".join(f"{l:>14}" for l in labels)
print(header)
print('-' * len(header))
for label, cat, key in metricas:
    vals = []
    for k in keys:
        v = results.get(k, {}).get(cat, {}).get(key, 'N/A')
        vals.append(f"{v:>14.2f}" if isinstance(v, float) else f"{str(v):>14}")
    print(f"{label:<25}" + "".join(vals))